# Predicting Airline Satisfaction: Baseline

Kaggle Playground Series S6E10. Binary probability of `satisfaction`,
scored by ROC AUC. This notebook is the first comparable model pair.
The fold definition, recipe, and promotion rule were written in
`docs/3_implementation_plan.md` before this run.

Private CPU kernel. Internet disabled. GPU is not used.


## 1. Environment


In [ ]:
import json
import platform
import shutil
import time
from pathlib import Path

import catboost
import lightgbm
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

print("python", platform.python_version())
print({
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
    "catboost": catboost.__version__,
})

## 2. Config


In [ ]:
SEED = 42
NOTEBOOK_VERSION = "v2"
TARGET = "satisfaction"
ID_COLUMN = "id"
ARRIVAL = "Arrival Delay in Minutes"
N_SPLITS = 5
PROMOTION_MARGIN = 0.0005
TREE_BUDGET = 500
LEARNING_RATE = 0.05

CATEGORICAL_FEATURES = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
]


def find_data_dir() -> Path:
    """Locate competition files without assuming one mount path.

    Returns:
        Directory that contains train.csv.

    Raises:
        FileNotFoundError: If train.csv is not on a tried path.
    """
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e10"),
        Path("/kaggle/input/playground-series-s6e10"),
        Path("../data"),
        Path("data"),
    ]
    for cand in candidates:
        if (cand / "train.csv").exists():
            return cand
    mount = Path("/kaggle/input")
    if mount.exists():
        hits = sorted(mount.rglob("train.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("train.csv not found in any known location")


DATA_DIR = find_data_dir()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("predictions")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA_DIR", DATA_DIR.resolve())
print("OUT", OUT.resolve())
print(
    "F1 = StratifiedKFold(5, shuffle=True, random_state=42). "
    f"Promote LightGBM only if OOF AUC is higher and the mean paired "
    f"fold gap is at least {PROMOTION_MARGIN}."
)

## 3. Data


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
y = train[TARGET].astype(int)
feature_columns = [c for c in train.columns if c not in (ID_COLUMN, TARGET)]
assert feature_columns == [c for c in test.columns if c != ID_COLUMN]
assert list(sample.columns) == [ID_COLUMN, TARGET]
print(train.shape, test.shape, "positive", int(y.sum()))


## 4. Fold-safe recipe

The missingness indicator uses the raw mask. The arrival-delay median
is fit on the training rows of the current fold only.


In [ ]:
def prepare_fold(
    train_frame: pd.DataFrame,
    apply_frame: pd.DataFrame,
    fit_index: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, float]:
    """Apply the shared recipe using one fold's training rows.

    Args:
        train_frame: Full training frame, raw columns only.
        apply_frame: Validation rows or the test frame.
        fit_index: Positions in train_frame used to fit the median.

    Returns:
        Transformed fit rows only, transformed apply_frame, and the median.
    """
    fit_rows = train_frame.iloc[fit_index]
    median = float(fit_rows[ARRIVAL].median())

    def transform(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame[feature_columns].copy()
        out["arrival_delay_missing"] = out[ARRIVAL].isna().astype(np.int8)
        out[ARRIVAL] = out[ARRIVAL].fillna(median)
        return out

    return transform(fit_rows), transform(apply_frame), median

## 5. Models


In [ ]:
def fit_catboost(x_fit: pd.DataFrame, y_fit: pd.Series) -> CatBoostClassifier:
    """Fit the predeclared CatBoost baseline.

    Args:
        x_fit: Training-fold features.
        y_fit: Training-fold labels.

    Returns:
        A fitted classifier.
    """
    model = CatBoostClassifier(
        iterations=TREE_BUDGET,
        learning_rate=LEARNING_RATE,
        depth=6,
        loss_function="Logloss",
        task_type="CPU",
        random_seed=SEED,
        verbose=100,
        allow_writing_files=False,
    )
    model.fit(x_fit, y_fit, cat_features=CATEGORICAL_FEATURES)
    return model


def fit_lightgbm(x_fit: pd.DataFrame, y_fit: pd.Series) -> LGBMClassifier:
    """Fit the predeclared LightGBM challenger.

    Args:
        x_fit: Training-fold features.
        y_fit: Training-fold labels.

    Returns:
        A fitted classifier.
    """
    frame = x_fit.copy()
    for col in CATEGORICAL_FEATURES:
        frame[col] = frame[col].astype("category")
    model = LGBMClassifier(
        n_estimators=TREE_BUDGET,
        learning_rate=LEARNING_RATE,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(frame, y_fit, categorical_feature=CATEGORICAL_FEATURES)
    return model


def predict(model, frame: pd.DataFrame) -> np.ndarray:
    """Predict the positive-class probability.

    Args:
        model: A fitted CatBoost or LightGBM classifier.
        frame: Features in the training column order.

    Returns:
        A one-dimensional probability vector.
    """
    if isinstance(model, LGBMClassifier):
        frame = frame.copy()
        for col in CATEGORICAL_FEATURES:
            frame[col] = frame[col].astype("category")
    return model.predict_proba(frame)[:, 1]


## 6. Cross-validation


In [ ]:
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = {
    "catboost": np.zeros(len(train), dtype=np.float64),
    "lightgbm": np.zeros(len(train), dtype=np.float64),
}
test_pred = {
    "catboost": np.zeros(len(test), dtype=np.float64),
    "lightgbm": np.zeros(len(test), dtype=np.float64),
}
fold_auc = {name: [] for name in oof}
timings = {name: [] for name in oof}

for fold_id, (fit_idx, val_idx) in enumerate(folds.split(train, y), start=1):
    x_fit, x_val, median = prepare_fold(train, train.iloc[val_idx], fit_idx)
    _, x_test, _ = prepare_fold(train, test, fit_idx)
    y_fit = y.iloc[fit_idx]
    y_val = y.iloc[val_idx]
    print(f"fold {fold_id} median {median:.4f} val rows {len(val_idx)}")
    for name, fitter in ("catboost", fit_catboost), ("lightgbm", fit_lightgbm):
        started = time.perf_counter()
        model = fitter(x_fit, y_fit)
        val_hat = predict(model, x_val)
        test_hat = predict(model, x_test)
        elapsed = time.perf_counter() - started
        oof[name][val_idx] = val_hat
        test_pred[name] += test_hat / N_SPLITS
        score = float(roc_auc_score(y_val, val_hat))
        fold_auc[name].append(score)
        timings[name].append(elapsed)
        print(f"  {name} fold AUC {score:.6f} seconds {elapsed:.1f}")

for name in oof:
    np.save(OUT / f"oof_{name}.npy", oof[name])
    np.save(OUT / f"test_{name}.npy", test_pred[name])
    print(
        name,
        "OOF", f"{roc_auc_score(y, oof[name]):.6f}",
        "fold mean", f"{np.mean(fold_auc[name]):.6f}",
        "fold std", f"{np.std(fold_auc[name]):.6f}",
        "seconds", f"{sum(timings[name]):.1f}",
    )


## 7. Promotion gate


In [ ]:
cat_auc = float(roc_auc_score(y, oof["catboost"]))
lgb_auc = float(roc_auc_score(y, oof["lightgbm"]))
paired = np.asarray(fold_auc["lightgbm"]) - np.asarray(fold_auc["catboost"])
mean_gap = float(paired.mean())
promote = (lgb_auc > cat_auc) and (mean_gap >= PROMOTION_MARGIN)
champion = "lightgbm" if promote else "catboost"
print(
    f"catboost {cat_auc:.6f} lightgbm {lgb_auc:.6f} "
    f"mean paired gap {mean_gap:.6f} promote {promote}"
)
print("champion", champion)
print("NOTEBOOK_VERSION", NOTEBOOK_VERSION)

chosen = test_pred[champion]
assert np.isfinite(chosen).all()
assert ((chosen >= 0.0) & (chosen <= 1.0)).all()
submission = pd.DataFrame({
    ID_COLUMN: test[ID_COLUMN].to_numpy(),
    TARGET: chosen,
})
submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)
summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "fold": "F1",
    "catboost_oof_auc": cat_auc,
    "lightgbm_oof_auc": lgb_auc,
    "mean_paired_gap": mean_gap,
    "promotion_margin": PROMOTION_MARGIN,
    "champion": champion,
    "fold_auc": fold_auc,
    "seconds": {name: sum(values) for name, values in timings.items()},
}
(OUT / "baseline_summary.json").write_text(json.dumps(summary, indent=2))
print("wrote", submission_path.name, len(submission))


## 8. Self-export

Last cell. Copies the executed notebook, which holds every output above
and none of this cell's own output.


In [ ]:
source = Path("/kaggle/working/__notebook__.ipynb")
if source.exists():
    dest = Path("/kaggle/working/executed_notebook.ipynb")
    shutil.copy(source, dest)
    exported = json.loads(dest.read_text())
    with_outputs = sum(1 for cell in exported["cells"] if cell.get("outputs"))
    print(
        f"self-export: {dest.name} — {len(exported['cells'])} cells, "
        f"{with_outputs} with outputs, {dest.stat().st_size / 1024:.0f} KB"
    )
else:
    print("no __notebook__.ipynb — self-export skipped")
